## Nettoyage des datasets

1. Import des bibliothèques

In [40]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print("Librairies chargées ")

Librairies chargées 


2. Chargement des datasets

In [41]:
data_path = Path.cwd().parent / "data" / "raw"
df1 = pd.read_csv(data_path / "data.csv", encoding="latin1")
df2 = pd.read_csv(data_path / "Train.csv", encoding="latin1")

print("Datasets chargés ✔")

Datasets chargés ✔


# Chemins du projet

In [42]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "bronze"
BRONZE_DIR = DATA_DIR / "silver"

BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print("RAW :", RAW_DIR.resolve())
print("BRONZE :", BRONZE_DIR.resolve())

RAW : C:\Users\Wiam Ougga\Downloads\ecommerce-analytics-project\data\bronze
BRONZE : C:\Users\Wiam Ougga\Downloads\ecommerce-analytics-project\data\silver


## DATASET 1 : data.csv

# Avant nettoyage 

In [43]:
print("=" * 60)
print("DIMENSIONS INITIALES")
print(df1.shape)

print("\nAPERÇU")
print(df1.head())

DIMENSIONS INITIALES
(541909, 8)

APERÇU
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  


In [44]:
print("=" * 60)
print("VALEURS MANQUANTES")
print(df1.isnull().sum())

print("\nTYPES DES COLONNES")
print(df1.dtypes)

print("\nDOUBLONS")
print(df1.duplicated().sum())

print("\nQUALITÉ MÉTIER")
print("Quantités <= 0 :", (df1["Quantity"] <= 0).sum())
print("Prix <= 0 :", (df1["UnitPrice"] <= 0).sum())
print("Factures annulées :", df1["InvoiceNo"].astype(str).str.startswith("C").sum())

VALEURS MANQUANTES
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

TYPES DES COLONNES
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

DOUBLONS
5268

QUALITÉ MÉTIER
Quantités <= 0 : 10624
Prix <= 0 : 2517
Factures annulées : 9288


In [45]:
#Copie du dataset
df1_clean = df1.copy()

In [46]:
#Suppression des doublons
avant = df1_clean.shape[0]

df1_clean = df1_clean.drop_duplicates()

apres = df1_clean.shape[0]

print("=" * 60)
print("DOUBLONS")
print("Supprimés :", avant - apres)

DOUBLONS
Supprimés : 5268


In [47]:
# Suppression des valeurs manquantes
avant = df1_clean.shape[0]

df1_clean = df1_clean.dropna(subset=["CustomerID", "Description"])

apres = df1_clean.shape[0]

print("=" * 60)
print("VALEURS MANQUANTES")
print("Lignes supprimées :", avant - apres)

VALEURS MANQUANTES
Lignes supprimées : 135037


In [48]:
#Suppression des factures annulées
avant = df1_clean.shape[0]

df1_clean = df1_clean[~df1_clean["InvoiceNo"].astype(str).str.startswith("C")]

apres = df1_clean.shape[0]

print("=" * 60)
print("FACTURES ANNULÉES")
print("Lignes supprimées :", avant - apres)

FACTURES ANNULÉES
Lignes supprimées : 8872


In [49]:
#Suppression des quantités invalides
avant = df1_clean.shape[0]

df1_clean = df1_clean[df1_clean["Quantity"] > 0]

apres = df1_clean.shape[0]

print("=" * 60)
print("QUANTITY")
print("Lignes supprimées :", avant - apres)

QUANTITY
Lignes supprimées : 0


In [50]:
# Suppression des prix invalides
avant = df1_clean.shape[0]

df1_clean = df1_clean[df1_clean["UnitPrice"] > 0]

apres = df1_clean.shape[0]

print("=" * 60)
print("UNIT PRICE")
print("Lignes supprimées :", avant - apres)

UNIT PRICE
Lignes supprimées : 40


In [51]:
# Conversion de dates
df1_clean["InvoiceDate"] = pd.to_datetime(df1_clean["InvoiceDate"], errors="coerce")

avant = df1_clean.shape[0]

df1_clean = df1_clean.dropna(subset=["InvoiceDate"])

apres = df1_clean.shape[0]

print("=" * 60)
print("DATES")
print("Lignes supprimées :", avant - apres)

DATES
Lignes supprimées : 0


In [52]:
# Reset index
df1_clean = df1_clean.reset_index(drop=True)

In [53]:
# Résumé final
print("=" * 60)
print("RÉSUMÉ FINAL")
print("=" * 60)

print("Shape initial :", df1.shape)
print("Shape final   :", df1_clean.shape)
print("Types final   :", df1_clean.dtypes)
print("Lignes supprimées :", df1.shape[0] - df1_clean.shape[0])

print("\nValeurs manquantes restantes :")
print(df1_clean.isnull().sum())

RÉSUMÉ FINAL
Shape initial : (541909, 8)
Shape final   : (392692, 8)
Types final   : InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object
Lignes supprimées : 149217

Valeurs manquantes restantes :
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


Sauvegarde dans data/bronze

In [54]:
output_path = BRONZE_DIR / "online_retail_clean.csv"

df1_clean.to_csv(output_path, index=False)

print("=" * 60)
print("FICHIER SAUVEGARDÉ")
print(output_path.resolve())

FICHIER SAUVEGARDÉ
C:\Users\Wiam Ougga\Downloads\ecommerce-analytics-project\data\silver\online_retail_clean.csv


## DATASET 2 : Train.csv

# Avant nettoyage

In [55]:
print("=" * 60)
print("DIMENSIONS INITIALES df2")
print(df2.shape)

print("\nAPERÇU")
print(df2.head())

DIMENSIONS INITIALES df2
(10999, 12)

APERÇU
   ï»¿ID Warehouse_block Mode_of_Shipment  Customer_care_calls  \
0      1               D           Flight                    4   
1      2               F           Flight                    4   
2      3               A           Flight                    2   
3      4               B           Flight                    3   
4      5               C           Flight                    2   

   Customer_rating  Cost_of_the_Product  Prior_purchases Product_importance  \
0                2                  177                3                low   
1                5                  216                2                low   
2                2                  183                4                low   
3                3                  176                4             medium   
4                2                  184                3             medium   

  Gender  Discount_offered  Weight_in_gms  Reached.on.Time_Y.N  
0      F          

In [56]:
print("=" * 60)
print("INFO GÉNÉRALE")
print(df2.info())

print("\nVALEURS MANQUANTES")
print(df2.isnull().sum())

print("\n% MANQUANTS")
print((df2.isnull().sum() / len(df2)) * 100)

print("\nDOUBLONS")
print(df2.duplicated().sum())

print("\nSTATISTIQUES")
print(df2.describe())

INFO GÉNÉRALE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10999 entries, 0 to 10998
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ï»¿ID                10999 non-null  int64 
 1   Warehouse_block      10999 non-null  object
 2   Mode_of_Shipment     10999 non-null  object
 3   Customer_care_calls  10999 non-null  int64 
 4   Customer_rating      10999 non-null  int64 
 5   Cost_of_the_Product  10999 non-null  int64 
 6   Prior_purchases      10999 non-null  int64 
 7   Product_importance   10999 non-null  object
 8   Gender               10999 non-null  object
 9   Discount_offered     10999 non-null  int64 
 10  Weight_in_gms        10999 non-null  int64 
 11  Reached.on.Time_Y.N  10999 non-null  int64 
dtypes: int64(8), object(4)
memory usage: 1.0+ MB
None

VALEURS MANQUANTES
ï»¿ID                  0
Warehouse_block        0
Mode_of_Shipment       0
Customer_care_calls    0
Customer_rating 

In [57]:
#Copie du dataset 
df2_clean = df2.copy()

# Nettoyage

In [58]:
# Suppression des doublons
avant = df2_clean.shape[0]

df2_clean = df2_clean.drop_duplicates()

apres = df2_clean.shape[0]

print("=" * 60)
print("DOUBLONS")
print("Supprimés :", avant - apres)

DOUBLONS
Supprimés : 0


In [59]:
# TRAITEMENT VALEURS MANQUANTES
print("=" * 60)
print("VALEURS MANQUANTES")

for col in df2_clean.columns:
    missing = df2_clean[col].isnull().sum()

    if missing > 0:
        print(f"\nColonne: {col}")
        print("Avant:", missing)

        if df2_clean[col].dtype in ["int64", "float64"]:
            value = df2_clean[col].median()
            df2_clean[col] = df2_clean[col].fillna(value)
            print("Médiane:", value)

        else:
            value = df2_clean[col].mode()[0]
            df2_clean[col] = df2_clean[col].fillna(value)
            print("Mode:", value)

        print("Après:", df2_clean[col].isnull().sum())

VALEURS MANQUANTES


In [60]:
# Conversion des dates
for col in df2_clean.columns:
    if "date" in col.lower():
        print("=" * 60)
        print("Conversion:", col)

        df2_clean[col] = pd.to_datetime(df2_clean[col], errors="coerce")

        print("Type:", df2_clean[col].dtype)

In [61]:
# Vérification finale
print("=" * 60)
print("RÉSULTAT FINAL")

print("Shape:", df2_clean.shape)

print("\nMissing values:")
print(df2_clean.isnull().sum())

print("\nAperçu:")
print(df2_clean.head())

RÉSULTAT FINAL
Shape: (10999, 12)

Missing values:
ï»¿ID                  0
Warehouse_block        0
Mode_of_Shipment       0
Customer_care_calls    0
Customer_rating        0
Cost_of_the_Product    0
Prior_purchases        0
Product_importance     0
Gender                 0
Discount_offered       0
Weight_in_gms          0
Reached.on.Time_Y.N    0
dtype: int64

Aperçu:
   ï»¿ID Warehouse_block Mode_of_Shipment  Customer_care_calls  \
0      1               D           Flight                    4   
1      2               F           Flight                    4   
2      3               A           Flight                    2   
3      4               B           Flight                    3   
4      5               C           Flight                    2   

   Customer_rating  Cost_of_the_Product  Prior_purchases Product_importance  \
0                2                  177                3                low   
1                5                  216                2                

In [63]:
#Sauvegarde dans data/bronze

In [64]:
output_path = BRONZE_DIR / "train_clean.csv"

df2_clean.to_csv(output_path, index=False)

print("=" * 60)
print("FICHIER SAUVEGARDÉ")
print(output_path.resolve())

FICHIER SAUVEGARDÉ
C:\Users\Wiam Ougga\Downloads\ecommerce-analytics-project\data\silver\train_clean.csv
